# Bab 2: Pemodelan & Evaluasi Lengkap

Dokumen ini berisi proses pelatihan model Random Forest Classifier dan Support Vector Machine (Linear SVM) serta perhitungan metrik evaluasi lengkap (Accuracy, Confusion Matrix, ROC-AUC).

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

print("[+] Library pemodelan berhasil dimuat!")

## 1. Membaca Dataset & Encoding

In [ ]:
df = pd.read_csv('../data/processed/hand_landmarks_data_clean.csv')
X = df.drop('label', axis=1)
y = df['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)
classes = le.classes_

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"Train Size: {X_train.shape[0]}, Test Size: {X_test.shape[0]}")

## 2. Melatih Model 1: Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%")

## 3. Melatih Model 2: Support Vector Machine (Linear SVM)

In [ ]:
svm_model = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
print(f"Linear SVM Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%")

## 4. Evaluasi Metrik: Confusion Matrix & ROC Curve

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix - SVM Model Terbaik')
plt.show()

In [ ]:
# Perhitungan ROC Curve Micro-Average
y_test_bin = label_binarize(y_test, classes=range(len(classes)))
y_score_rf = rf_model.predict_proba(X_test)
y_score_svm = svm_model.predict_proba(X_test)

fpr_rf, tpr_rf, _ = roc_curve(y_test_bin.ravel(), y_score_rf.ravel())
fpr_svm, tpr_svm, _ = roc_curve(y_test_bin.ravel(), y_score_svm.ravel())

plt.figure(figsize=(8, 6))
plt.plot(fpr_svm, tpr_svm, color='darkorange', label=f'SVM (AUC = {auc(fpr_svm, tpr_svm):.4f})')
plt.plot(fpr_rf, tpr_rf, color='navy', label=f'RF (AUC = {auc(fpr_rf, tpr_rf):.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Kurva ROC - Perbandingan Model')
plt.legend()
plt.show()